In [ ]:
# Quantum imports
import qutip

# Linalg imports
import numpy as np
from numpy.random import default_rng

# Plotting imports
import matplotlib.pyplot as plt
import seaborn as sns

## Convergence in Entropy

Here, we increase the size of the generated matrices and look at the variance in their spectrum by computing the entropy. We want to see two things:

1. When does increasing matrix size result in no changes to the spectrum
2. How much variance exists in a certain matrix size.

In doing so, we can learn better how to choose maximal observables for the reservoir computations

In [ ]:
densities = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
ensembles = 100
measure_states = [1, 2, 3, 4, 5, 6, 7, 8, 9]
entropies = {}
spectra = {}

for density in densities:
    spectra[density] = {}
    entropies[density] = {}
    for state in measure_states:
        entropies[density][state] = []
        spectra[density][state] = []
        for _ in range(ensembles):
            non_gue_matrix = qutip.rand_herm(
                2**state, 
                density, 
                dims=[[2] * state, [2] * state]
            )      

            entropies[density][state].append(qutip.entropy_vn(non_gue_matrix, 2))
            spectra[density][state].append(np.linalg.eigh(non_gue_matrix).eigenvalues)


In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(8.4, 8))

densities = [0.1, 0.3, 0.5, 0.6, 0.8, 1.0]
measure_states = [1, 5, 9]

for i, density in enumerate(densities):
    row = i // 3
    col = i % 3
    for state in measure_states:
        sns.kdeplot(
            np.concatenate(spectra[density][state]), 
            fill=True, 
            label=state, 
            ax=ax[row, col]
        )

ax[1, 0].set_xlabel(r"$\lambda_{i}$")
ax[1, 1].set_xlabel(r"$\lambda_{i}$")
ax[1, 2].set_xlabel(r"$\lambda_{i}$")

ax[0, 0].set_ylabel(r"$\rho\left(\lambda_{i}\right)$")
ax[1, 0].set_ylabel(r"$\rho\left(\lambda_{i}\right)$")

ax[1, 1].set_ylabel(None)
ax[1, 2].set_ylabel(None)
ax[0, 1].set_ylabel(None)
ax[0, 2].set_ylabel(None)

ax[0, 0].legend(title="Dimension")

plt.legend()
plt.savefig("spectra.pdf")
plt.show()

## Convergence in Expectation Values

Now we examine what the range of observables are when applied to random density matrices.

In [ ]:
densities = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
ensembles = 50
measure_states = [1, 2, 3, 4, 5, 6, 7, 8, 9]

states = [qutip.rand_dm(2 ** 9, 0.2, dims=[[2] * 9, [2] * 9]) for _ in range(10)]

observables = {}

for density in densities:
    observables[density] = {}

    for state in measure_states:
        observables[density][state] = []
        for _ in range(ensembles):
            non_gue_matrix = qutip.rand_herm(
                2**state, 
                density, 
                dims=[[2] * state, [2] * state]
            )
            for dm in states:
                if state < 9:
                    rng = default_rng()
                    indices = rng.choice(9, size=state, replace=False)
                    dm = dm.ptrace(indices)
                observables[density][state].append(
                    qutip.expect(
                        dm, non_gue_matrix
                    )
                )


In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(8.4, 8))

densities = [0.1, 0.3, 0.5, 0.6, 0.8, 1.0]
measure_states = [1, 5, 9]

for i, density in enumerate(densities):
    row = i // 3
    col = i % 3
    for state in measure_states:
        sns.kdeplot(
            observables[density][state],
            fill=True, 
            label=state, 
            ax=ax[row, col]
        )
    ax[row, col].set_title(density)

ax[1, 0].set_xlabel(r"$\langle\mathcal{O}\rangle$")
ax[1, 1].set_xlabel(r"$\langle\mathcal{O}\rangle$")
ax[1, 2].set_xlabel(r"$\langle\mathcal{O}\rangle$")

ax[0, 0].set_ylabel(r"$\rho\left(\langle\mathcal{O}\rangle\right)$")
ax[1, 0].set_ylabel(r"$\rho\left(\langle\mathcal{O}\rangle\right)$")

ax[1, 1].set_ylabel(None)
ax[1, 2].set_ylabel(None)
ax[0, 1].set_ylabel(None)
ax[0, 2].set_ylabel(None)

ax[0, 0].legend(title="Dimension")

plt.savefig("observables.pdf")
plt.show()